# Node Documentatie – Boodschappen Agent Workflow

Dit document beschrijft per node de **inputs**, **outputs**, **validaties/vergelijkingen**, **terugkoppelingen (loops)**, en welke **LLM, tools/APIs of regel-gebaseerde kennis** nodig zijn.

---

## Legenda

| Symbool | Betekenis |
|---|---|
| 🧠 LLM | Generatieve taalmodel actie (bv. GPT-4o, Claude, Gemini – vergelijking nodig) |
| ⚙️ Regel | Deterministische regel-gebaseerde code |
| 🔧 Tool/API | Externe API of library |
| 📚 Vector DB | Vector database (regel-gebaseerde kennis / RAG) |
| 🔁 Loop terug | Stap terug in proces (heroverweging) |

---

## Node 1 – Gebruiker Input

| Veld | Beschrijving |
|---|---|
| **Input** | Vrije tekst van gebruiker (bv. "Ik wil deze week pasta, salade en ontbijt voor 4 personen, budget €40") |
| **Output** | Ruwe gestructureerde intent (gerechten, personen, budget, voorkeuren, dieet) |
| **LLM nodig?** | 🧠 **Ja** – intent-extractie en normalisatie van vrije tekst → JSON schema |
| **Tools/APIs** | Geen externe API; alleen LLM-call |
| **Regel/Vector** | 📚 Vector DB lookup voor *gebruikersprofiel* (eerdere voorkeuren, allergieën, vaste boodschappen) |
| **Validatie** | Schema-validatie (Pydantic/JSON-schema) op output van LLM |
| **Loop terug** | n.v.t. – startpunt |

---

## Node 2 – Locatie Bepalen

| Veld | Beschrijving |
|---|---|
| **Input** | Adres / postcode / GPS van gebruiker, of opgeslagen locatie uit profiel |
| **Output** | Gevalideerde coördinaten (lat/lon) + postcode + actieradius (km) |
| **LLM nodig?** | ❌ Nee – puur deterministisch |
| **Tools/APIs** | 🔧 **Google Maps Geocoding API** (adres → coördinaten) |
| **Regel/Vector** | 📚 Vector DB: ophalen *opgeslagen locatie* van gebruiker (zie pijl "Locatie ophalen" terug naar Node 2) |
| **Validatie** | - Postcode-format (NL: `1234 AB`)<br>- Coördinaten binnen NL bounding box<br>- Actieradius > 0 en ≤ max (bv. 15 km) |
| **Loop terug** | 🔁 Vanuit Node 4 ("Locatie aanwijzen") als er **geen winkels gevonden** worden binnen radius → radius vergroten of nieuwe locatie vragen |

---

## Node 3 – Boodschappen Lijst

| Veld | Beschrijving |
|---|---|
| **Input** | Gerechten/intent uit Node 1 |
| **Output** | Gestructureerde boodschappenlijst: `[{product, hoeveelheid, eenheid, categorie, substitueerbaar}]` |
| **LLM nodig?** | 🧠 **Ja** – recept-naar-ingrediënten decompositie + hoeveelheden schalen op aantal personen |
| **Tools/APIs** | Optioneel: receptendatabase API (bv. Spoonacular) |
| **Regel/Vector** | 📚 Vector DB met *standaard ingrediënten per gerecht* en *eenheidsconversies* (g, ml, stuks) |
| **Validatie** | - Eenheden gestandaardiseerd<br>- Geen dubbele items (samenvoegen)<br>- Allergie-check tegen profiel ⚙️ regel |
| **Loop terug** | 🔁 Naar Node 1 als items onduidelijk of conflict met dieet/budget |

---

## Node 4 – Winkels Zoeken

| Veld | Beschrijving |
|---|---|
| **Input** | Coördinaten + radius (Node 2), boodschappenlijst (Node 3) |
| **Output** | Lijst van supermarkten binnen radius: `[{naam, keten, adres, afstand, openingstijden}]` |
| **LLM nodig?** | ❌ Nee |
| **Tools/APIs** | 🔧 **Google Maps Places API** (`type=supermarket`) + reisafstand via Distance Matrix API |
| **Regel/Vector** | ⚙️ Filter op ondersteunde ketens (AH, Jumbo, Lidl, Plus – afhankelijk van wat `supermarktconnector` ondersteunt) |
| **Validatie** | - Min. 1 winkel gevonden<br>- Winkel is open op gewenste dag/tijd<br>- Keten matcht supported list |
| **Loop terug** | 🔁 **"Locatie aanwijzen"** terug naar Node 2 als 0 winkels gevonden → radius vergroten / alternatieve locatie |

---

## Node 5 – Prijzen Ophalen

| Veld | Beschrijving |
|---|---|
| **Input** | Boodschappenlijst + gevonden winkelketens |
| **Output** | Prijs per product per keten: `[{product, keten, prijs, aanbieding, beschikbaar}]` |
| **LLM nodig?** | 🧠 **Optioneel** – alleen voor *product matching* (fuzzy match "tomaat" → "Tomaten los per stuk") als regels falen |
| **Tools/APIs** | 🔧 **`bartmachielsen/supermarktconnector`** (GitHub) – AH & Jumbo product/prijs ophalen |
| **Regel/Vector** | 📚 Vector DB voor *product embeddings* → semantische match tussen boodschappen-item en winkel-SKU<br>⚙️ Regel: exacte match eerst, dan fuzzy, dan vector, dan LLM |
| **Validatie** | - Prijs > 0 en < plafond (anti-outlier)<br>- Eenheid komt overeen (€/kg vs €/stuk)<br>- Beschikbaarheidsvlag |
| **Loop terug** | 🔁 Naar Node 3 als veel producten niet matchen → lijst herformuleren met algemenere termen |

---

## Node 6 – Vergelijking Supermarkt

| Veld | Beschrijving |
|---|---|
| **Input** | Prijsmatrix uit Node 5 + winkellijst uit Node 4 |
| **Output** | Ranking van winkels/combinaties: `[{winkel(s), totaalprijs, afstand, dekking%}]` |
| **LLM nodig?** | 🧠 **Optioneel** – LLM voor *uitleg* van het advies (rationale), niet voor de berekening zelf. **LLM-vergelijking nodig**: test GPT-4o vs Claude vs Gemini op kwaliteit van uitleg + consistentie |
| **Tools/APIs** | Geen |
| **Regel/Vector** | ⚙️ **Regel-gebaseerd**: gewogen scoring<br>`score = w1·totaalprijs + w2·afstand + w3·(1 - dekking) + w4·reistijd` |
| **Validatie** | - Dekking ≥ drempel (bv. 80% van lijst)<br>- Vergelijking single-store vs multi-store<br>- Budget-check tegen Node 1 |
| **Loop terug** | 🔁 Naar Node 4 als geen enkele winkel(combinatie) voldoet aan dekking/budget |

---

## Node 7 – Route Planning

| Veld | Beschrijving |
|---|---|
| **Input** | Gekozen winkel(s) uit Node 6 + locatie gebruiker |
| **Output** | Geoptimaliseerde route (volgorde, reistijd, modaliteit: lopen/fiets/auto) |
| **LLM nodig?** | ❌ Nee |
| **Tools/APIs** | 🔧 **Google Maps Directions API** + **Routes API** (TSP voor multi-stop) |
| **Regel/Vector** | ⚙️ Regel: bij 1 winkel → directe route; bij N>1 → nearest-neighbor of OR-tools TSP |
| **Validatie** | - Totale reistijd binnen redelijke grens<br>- Route haalbaar binnen openingstijden<br>- Modaliteit matcht voorkeur gebruiker |
| **Loop terug** | 🔁 Naar Node 6 als reistijd > acceptabel → herwegen scoring (meer gewicht op afstand) |

---

## Node 8 – Supermarkt Advies

| Veld | Beschrijving |
|---|---|
| **Input** | Ranking (Node 6) + route (Node 7) + boodschappenlijst (Node 3) |
| **Output** | Mens-leesbaar advies: aanbevolen winkel(s), totaalprijs, route, motivatie, alternatieven |
| **LLM nodig?** | 🧠 **Ja** – natuurlijke taal-generatie van advies. **LLM-vergelijking nodig** (zelfde modellen als Node 6) op: toon, beknoptheid, hallucinatie-rate |
| **Tools/APIs** | Geen |
| **Regel/Vector** | 📚 Vector DB: persoonlijke voorkeuren voor toon/uitgebreidheid |
| **Validatie** | - Guardrail: geen prijzen/winkels in tekst die niet in input zitten (anti-hallucinatie)<br>- ⚙️ Regel: structured output validation |
| **Loop terug** | 🔁 Naar Node 6 als gebruiker advies afwijst (feedback-loop via Node 9) |

---

## Node 9 – Gebruiker Updaten

| Veld | Beschrijving |
|---|---|
| **Input** | Advies (Node 8) + gebruikersfeedback (geaccepteerd/afgewezen/aangepast) |
| **Output** | Geüpdatete vector DB (voorkeuren, geaccepteerde winkels, prijsgeschiedenis) |
| **LLM nodig?** | 🧠 **Optioneel** – LLM om vrije-tekst feedback te interpreteren naar profielupdates |
| **Tools/APIs** | Geen externe API |
| **Regel/Vector** | 📚 **Schrijven naar Vector DB**: embeddings van geaccepteerde producten, winkels, gerechten |
| **Validatie** | - Geen PII in embeddings<br>- Versie-tag op profielupdate |
| **Loop terug** | 🔁 Volledige feedback-loop terug naar Node 1 voor volgende sessie (heroverweging) |

---

## Overzicht Loops (stappen terug)

| Vanuit | Naar | Trigger |
|---|---|---|
| Node 4 | Node 2 | Geen winkels binnen radius → "Locatie aanwijzen" |
| Node 2 | (DB) | Bestaande locatie ophalen uit profiel |
| Node 5 | Node 3 | Te veel onmatchbare producten |
| Node 6 | Node 4 | Onvoldoende dekking of budget overschreden |
| Node 7 | Node 6 | Route te lang → scoring herwegen |
| Node 8 | Node 6 | Advies afgewezen door guardrail of gebruiker |
| Node 9 | Node 1 | Volgende sessie / heroverweging |

---

## Samenvatting Tools, APIs en Kennisbronnen

| Bron | Type | Gebruikt in nodes |
|---|---|---|
| **Google Maps Geocoding API** | 🔧 API | 2 |
| **Google Maps Places API** | 🔧 API | 4 |
| **Google Maps Distance Matrix / Directions API** | 🔧 API | 4, 7 |
| **`bartmachielsen/supermarktconnector`** | 🔧 GitHub library | 5 |
| **Vector Database** (bv. Pinecone / Weaviate / pgvector) | 📚 RAG | 1, 3, 5, 8, 9 |
| **LLM (vergelijking: GPT-4o / Claude / Gemini)** | 🧠 LLM | 1, 3, 5*, 6*, 8, 9* |
| **Regel-gebaseerde scoring & validatie** | ⚙️ Regel | 2, 3, 4, 5, 6, 7, 8 |

\* = optioneel / fallback

---

## LLM-vergelijking (te documenteren in evaluatie)

Voor nodes met 🧠 LLM moet vergeleken worden op:

1. **Accuratesse** – correcte intent-extractie / decompositie
2. **Hallucinatie-rate** – verzint het producten/prijzen?
3. **Latency** – respons-tijd
4. **Kosten** – €/1k tokens
5. **Consistentie** – zelfde input → zelfde output (temperatuur=0)
6. **Nederlandse taal** – kwaliteit NL-output

Modellen om te benchmarken: **GPT-4o**, **Claude Sonnet 4.5**, **Gemini 2.5 Pro**, **Llama 3.3 (lokaal)**.
